# Extended report

Short extensions on top of `replication_report.ipynb`.

**GPU benchmarks:** run `sbatch jobs/08_extensions_beauty.sbatch` and `08_extensions_sports.sbatch` (needs checkpoints on scratch).  
**This notebook:** reads `reports/` + `reports/extensions/` — works on laptop after you commit exported CSVs.


In [ ]:
# Optional: run GPU benchmark from notebook (Snellius GPU node only). Default: skip — use sbatch jobs instead.
RUN_GPU_BENCHMARK = False
BENCHMARK_CATEGORY = "Beauty"  # or Sports_and_Outdoors
BENCHMARK_SLUG = "beauty"      # or sports

if RUN_GPU_BENCHMARK:
    import os
    import subprocess
    import sys
    from pathlib import Path
    REPO = Path.cwd().resolve()
    if REPO.name == "notebooks":
        REPO = REPO.parent
    env = os.environ.copy()
    env.setdefault("OUTPUT_ROOT", str(Path(env.get("SCRATCH", "/home/scur1266/scratch")) / "cosette_marius/outputs"))
    cmd = [
        sys.executable,
        str(REPO / "scripts/benchmark_extensions.py"),
        "--category", BENCHMARK_CATEGORY,
        "--category-slug", BENCHMARK_SLUG,
        "--seed", "42",
        "--output-root", env["OUTPUT_ROOT"],
        "--results-dir", str(Path(env["OUTPUT_ROOT"]) / "results"),
    ]
    subprocess.run(cmd, cwd=REPO, env=env, check=True)
else:
    print("GPU benchmark skipped. Run: sbatch jobs/08_extensions_beauty.sbatch && sbatch jobs/08_extensions_sports.sbatch")


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.reporting.extension_loaders import (
    cosette_collision_df,
    extensions_dir,
    load_inference_benchmark,
    load_model_vocab,
    load_wall_clock,
    per_seed_spread,
    val_test_gap_table,
)

REPORTS = REPO_ROOT / "reports"
EXT = extensions_dir(REPO_ROOT)
RESULTS = REPORTS / "results"
METRICS = REPORTS / "metrics"

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


## 1. Inference time & GPU memory (test split)

In [ ]:
inf_df = load_inference_benchmark(EXT / "inference_benchmark.csv")
if inf_df.empty:
    print("No inference_benchmark.csv — run jobs 08_extensions_* on Snellius first.")
else:
    test_df = inf_df[inf_df["split"] == "test"].copy()
    display(test_df[["category", "method", "seed", "seconds", "peak_mem_gb", "n_params"]])
    for category in test_df["category"].unique():
        part = test_df[test_df["category"] == category]
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
        for ax, col, title in zip(axes, ["seconds", "peak_mem_gb"], ["Test inference time (s)", "Peak GPU memory (GB)"]):
            pivot = part.pivot(index="method", columns="seed", values=col)
            pivot.plot(kind="bar", ax=ax, legend=False)
            ax.set_title(f"{category}: {title}")
            ax.set_xlabel("")
        fig.tight_layout()
        plt.show()


## 2. Training wall-clock (5-seed jobs)

In [ ]:
wc = load_wall_clock(EXT / "wall_clock.json")
if not wc:
    print("No wall_clock.json — run jobs 08 (writes json even without GPU) or copy from Snellius.")
else:
    rows = []
    for cat, info in wc.items():
        if not isinstance(info, dict):
            continue
        for k, v in info.items():
            if k.endswith("_hours") and v is not None:
                rows.append({"category": cat, "job": k.replace("_hours", ""), "hours": v})
    display(pd.DataFrame(rows))


## 3. Val vs test gap (R@10)

In [ ]:
gap_df = val_test_gap_table(RESULTS, METRICS)
if gap_df.empty:
    print("Missing results or metrics CSVs.")
else:
    display(gap_df.round(3))
    fig, ax = plt.subplots(figsize=(8, 3.5))
    labels = [f"{r.category}\n{r.method}" for r in gap_df.itertuples()]
    x = range(len(labels))
    ax.bar(x, gap_df["gap_pp"], color="#4C72B0")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(list(x), labels, fontsize=8)
    ax.set_ylabel("val − test R@10 (pp)")
    ax.set_title("Validation higher than test → positive gap")
    fig.tight_layout()
    plt.show()


## 4. Model / vocab size

In [ ]:
mv = load_model_vocab(EXT / "model_vocab.json")
if mv:
    display(pd.DataFrame(mv).T)
else:
    display(pd.DataFrame({
        "Beauty": {"sasrec_vocab": 12103, "marius_code_vocab": 1026},
        "Sports_and_Outdoors": {"sasrec_vocab": 18359, "marius_code_vocab": 1026},
    }).T)


## 5. COSETTE collision rate

In [ ]:
col_df = cosette_collision_df(METRICS)
if col_df.empty:
    print("No COSETTE collision curves in reports/metrics/.")
else:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for name, grp in col_df.groupby(col_df["source"].str.replace("cosette_", "").str.replace(".csv", "")):
        grp = grp.sort_values("step")
        ax.plot(grp["step"], grp["value"], label=name, alpha=0.85)
    ax.set_xlabel("epoch")
    ax.set_ylabel("collision rate")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


## 6. Per-seed accuracy spread

In [ ]:
spread_df = per_seed_spread(RESULTS)
if spread_df.empty:
    print("No scores jsonl in reports/results/.")
else:
    display(spread_df.pivot_table(index="seed", columns=["category", "method"], values="R@10"))
    fig, ax = plt.subplots(figsize=(9, 4))
    for (cat, method), grp in spread_df.groupby(["category", "method"]):
        ax.plot(grp["seed"], grp["R@10"], marker="o", label=f"{cat} {method}")
    ax.set_xlabel("seed")
    ax.set_ylabel("test R@10 (%)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
